# English Graduation Projects sample

## Notebook overview

This notebook documents the **prompt-generation workflow** for the English graduation-project sample. It generates prompt files from the AI-free sample and then summarizes the resulting prompt/output corpus with simple word-count statistics.

**Main workflow**
1. Build prompt files from the AI-free source texts.
2. Run the generation step that writes the prompt outputs to Drive.
3. Summarize the resulting files to verify corpus size and distribution.

## Generating Prompts Using AI-Free Sample

### Code block 1 — Build prompt files from the AI-free source corpus

This block mounts Drive, initializes the OpenAI client, reads the AI-free graduation-project sample, and creates prompt files that can later be used for controlled AI generation.

In [ ]:
!pip install -q openai

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
import time
from pathlib import Path
from openai import OpenAI

# =========================
# SETTINGS
# =========================
os.environ["OPENAI_API_KEY"] = "${REDACTED_SECRET}"

client = OpenAI()

INPUT_FOLDER = "${PROJECT_DATA_PATH}"
OUTPUT_FOLDER = "${PROJECT_DATA_PATH}"

MODEL = "gpt-4.1-mini"
MAX_CHARS_PER_FILE = 45000   # increase if needed, but this helps control cost
SLEEP_BETWEEN_CALLS = 1.0    # seconds

input_path = Path(INPUT_FOLDER)
output_path = Path(OUTPUT_FOLDER)
output_path.mkdir(parents=True, exist_ok=True)

txt_files = sorted([p for p in input_path.iterdir() if p.is_file() and p.suffix.lower() == ".txt"])

if not txt_files:
    raise FileNotFoundError(f"No .txt files found in: {INPUT_FOLDER}")

def read_text_file(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return path.read_text(encoding=enc)
        except:
            pass
    raise ValueError(f"Could not read file: {path.name}")

def build_messages(file_name, file_text):
    truncated_text = file_text[:MAX_CHARS_PER_FILE]

    system_msg = """
You are helping with a research workflow.

Task:
Read a student's finished project/report text file and infer ONE realistic prompt that the student could have asked ChatGPT so that ChatGPT could produce a document like this as the output.

Important requirements for the prompt you write:
1. The prompt must sound like a real student request to ChatGPT.
2. It must ask ChatGPT to write/complete the project or report, not to analyze it.
3. It must capture the likely task, deliverable, structure, tone, level, and major content requirements.
4. It should be specific enough that a model could generate something similar in genre, scope, and style.
5. Do NOT quote or copy long passages from the file.
6. Do NOT mention that you were given the finished file.
7. Do NOT mention AI detection, reverse engineering, or research.
8. Output ONLY the prompt text, with no title, no bullets, no explanation, and no quotation marks unless naturally needed.
"""

    user_msg = f"""
File name: {file_name}

Finished student file content:
--------------------
{truncated_text}
--------------------

Write the single best student-style prompt.
"""
    return [
        {"role": "system", "content": system_msg.strip()},
        {"role": "user", "content": user_msg.strip()},
    ]

success = 0
failed = []

for i, file_path in enumerate(txt_files, start=1):
    try:
        file_text = read_text_file(file_path).strip()
        if not file_text:
            print(f"[{i}/{len(txt_files)}] Skipped empty file: {file_path.name}")
            failed.append((file_path.name, "Empty file"))
            continue

        response = client.responses.create(
            model=MODEL,
            input=build_messages(file_path.name, file_text)
        )

        prompt_text = response.output_text.strip()

        if not prompt_text:
            raise ValueError("Model returned empty output.")

        out_file = output_path / file_path.name
        out_file.write_text(prompt_text, encoding="utf-8")

        success += 1
        print(f"[{i}/{len(txt_files)}] Saved prompt: {out_file.name}")

        time.sleep(SLEEP_BETWEEN_CALLS)

    except Exception as e:
        print(f"[{i}/{len(txt_files)}] FAILED: {file_path.name} -> {e}")
        failed.append((file_path.name, str(e)))

print("\n=========================")
print(f"Done. Saved {success} prompt files to:")
print(OUTPUT_FOLDER)

if failed:
    print("\nFailed files:")
    for name, err in failed:
        print(f"- {name}: {err}")
else:
    print("\nNo failures.")

### Code block 2 — Generate outputs from the prepared prompt files

This block reads the prompt folder, sends the prompts to the model, and saves the resulting generated texts to Drive. Together with the previous cell, it forms the prompt-to-output generation pipeline.

In [ ]:
import os
!pip install -q openai

drive.mount('/content/drive', force_remount=False)

import re

# =========================
# SETTINGS
# =========================
os.environ["OPENAI_API_KEY"] = "${REDACTED_SECRET}"

client = OpenAI()

PROMPTS_FOLDER = "${PROJECT_DATA_PATH}"
OUTPUT_FOLDER  = "${PROJECT_DATA_PATH}"

MODEL = "gpt-4.1-mini"
SLEEP_BETWEEN_CALLS = 1.0

prompts_path = Path(PROMPTS_FOLDER)
output_path = Path(OUTPUT_FOLDER)
output_path.mkdir(parents=True, exist_ok=True)

prompt_files = sorted([p for p in prompts_path.iterdir() if p.is_file() and p.suffix.lower() == ".txt"])

if not prompt_files:
    raise FileNotFoundError(f"No .txt files found in: {PROMPTS_FOLDER}")

def read_text_file(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return path.read_text(encoding=enc)
        except:
            pass
    raise ValueError(f"Could not read file: {path.name}")

def clean_model_output(text):
    text = text.strip()

    # Remove markdown fences if model wraps output in them
    text = re.sub(r"^\s*```[a-zA-Z0-9_-]*\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)

    # Remove common intro lines at the beginning
    intro_patterns = [
        r"^\s*here(?:'s| is)\b.*?\n+",
        r"^\s*certainly\b.*?\n+",
        r"^\s*sure\b.*?\n+",
        r"^\s*yes\b.*?\n+",
        r"^\s*i can help\b.*?\n+",
        r"^\s*below is\b.*?\n+",
        r"^\s*let(?: me)?(?:'s| us)?\b.*?\n+",
        r"^\s*of course\b.*?\n+",
    ]
    changed = True
    while changed:
        changed = False
        for pat in intro_patterns:
            new_text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
            if new_text != text:
                text = new_text.strip()
                changed = True

    # Remove common ending lines at the end
    outro_patterns = [
        r"\n+\s*let me know if .*?$",
        r"\n+\s*if you(?:'d| would) like .*?$",
        r"\n+\s*feel free to .*?$",
        r"\n+\s*i can also .*?$",
        r"\n+\s*hope this helps.*?$",
        r"\n+\s*let me know whether .*?$",
        r"\n+\s*let me know if you need .*?$",
        r"\n+\s*you can ask me .*?$",
    ]
    changed = True
    while changed:
        changed = False
        for pat in outro_patterns:
            new_text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
            if new_text != text:
                text = new_text.strip()
                changed = True

    return text.strip()

success = 0
failed = []

SYSTEM_INSTRUCTION = """
You are writing the final deliverable only.

Return only the requested project/report/assignment content itself.
Do NOT add any introduction, explanation, acknowledgment, preface, summary note, or closing note.
Do NOT say things like:
- "Sure, here is..."
- "I can help with that"
- "Let me know if you want changes"
- "Hope this helps"

Output only the final document text.
"""

for i, file_path in enumerate(prompt_files, start=1):
    try:
        prompt_text = read_text_file(file_path).strip()

        if not prompt_text:
            print(f"[{i}/{len(prompt_files)}] Skipped empty prompt file: {file_path.name}")
            failed.append((file_path.name, "Empty prompt file"))
            continue

        response = client.responses.create(
            model=MODEL,
            input=[
                {"role": "system", "content": SYSTEM_INSTRUCTION.strip()},
                {"role": "user", "content": prompt_text}
            ]
        )

        generated_text = response.output_text.strip()

        if not generated_text:
            raise ValueError("Model returned empty output.")

        generated_text = clean_model_output(generated_text)

        if not generated_text:
            raise ValueError("Output became empty after cleanup.")

        out_file = output_path / file_path.name
        out_file.write_text(generated_text, encoding="utf-8")

        success += 1
        print(f"[{i}/{len(prompt_files)}] Saved generated file: {out_file.name}")

        time.sleep(SLEEP_BETWEEN_CALLS)

    except Exception as e:
        print(f"[{i}/{len(prompt_files)}] FAILED: {file_path.name} -> {e}")
        failed.append((file_path.name, str(e)))

print("\n=========================")
print(f"Done. Saved {success} generated files to:")
print(OUTPUT_FOLDER)

if failed:
    print("\nFailed files:")
    for name, err in failed:
        print(f"- {name}: {err}")
else:
    print("\nNo failures.")

### Code block 3 — Inspect the generated output files

This block performs a basic inspection of the generated TXT files and prints quick checks so the generation step can be validated before the corpus is summarized.

In [ ]:
drive.mount("/content/drive", force_remount=False)


# ====== CHANGE THIS ======
FOLDER = "${PROJECT_DATA_PATH}"
# =========================

folder_path = Path(FOLDER)

txt_files = sorted(
    [p for p in folder_path.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]
)

if not txt_files:
    raise FileNotFoundError(f"No .txt files found in: {FOLDER}")

for file_path in txt_files:
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            text = file_path.read_text(encoding=enc)
            break
        except:
            text = None
    if text is None:
        print(f"Could not read: {file_path.name}")
        continue

    cleaned = text.replace("###", "").replace("---", "")
    file_path.write_text(cleaned, encoding="utf-8")
    print(f"Cleaned: {file_path.name}")

print("Done.")

## Statistics

### Code block 4 — Compute corpus statistics for the generated outputs

This block summarizes word counts across the generated English graduation-project sample so the dataset can be checked for size and distribution.

In [ ]:
drive.mount("/content/drive", force_remount=False)

import os
import re
from collections import defaultdict

ROOT = "${PROJECT_DATA_PATH}"
BIN = 600


def count_words(text: str) -> int:
    return len(re.findall(r"[\w\u0600-\u06FF]+(?:['-][\w\u0600-\u06FF]+)?", text))


def bin_label(w: int) -> str:
    if w < BIN:
        return f"<{BIN}"
    lo = (w // BIN) * BIN
    hi = lo + BIN
    return f"{lo}-{hi}"


file_rows = []  # (subfolder, relpath, words)
total_words_all = 0

for root, _, files in os.walk(ROOT):
    for fn in files:
        if not fn.lower().endswith(".txt"):
            continue
        path = os.path.join(root, fn)
        rel = os.path.relpath(path, ROOT)
        subfolder = os.path.dirname(rel) if os.path.dirname(rel) else "."
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            txt = f.read()
        wc = count_words(txt)
        total_words_all += wc
        file_rows.append((subfolder, rel, wc))

if not file_rows:
    print("No .txt files found under:", ROOT)
else:
    # ---- files <300 ----
    lt300 = sorted(
        [r for r in file_rows if r[2] < 300], key=lambda x: (x[0], x[2], x[1])
    )
    print("FILES <300 WORDS (subfolder | words | file):")
    for sub, rel, wc in lt300:
        print(f"{sub} | {wc} | {rel}")

    # ---- per-subfolder bin distribution + per-subfolder word totals ----
    per_sub_counts = defaultdict(lambda: defaultdict(int))
    per_sub_files = defaultdict(int)
    per_sub_words = defaultdict(int)

    max_words = max(wc for _, _, wc in file_rows)
    labels = [f"<{BIN}"]
    hi = BIN
    while hi <= ((max_words // BIN) + 2) * BIN:
        labels.append(f"{hi}-{hi+BIN}")
        hi += BIN

    for sub, _, wc in file_rows:
        per_sub_files[sub] += 1
        per_sub_words[sub] += wc
        per_sub_counts[sub][bin_label(wc)] += 1

    print("\nPER-SUBFOLDER DISTRIBUTION (600-word bins):")
    for sub in sorted(per_sub_files.keys()):
        n = per_sub_files[sub]
        print(f"\n{sub}  (total files: {n} | total words: {per_sub_words[sub]})")
        for lab in labels:
            c = per_sub_counts[sub].get(lab, 0)
            pct = 100.0 * c / n
            if pct == 0:
                continue
            print(f"{lab}: {c} files ({pct:.2f}%)")

    # ---- word count per subfolder (requested) ----
    print(
        "\nWORD COUNT PER SUBFOLDER (subfolder | total words | files | avg words/file):"
    )
    for sub in sorted(per_sub_files.keys()):
        n = per_sub_files[sub]
        tw = per_sub_words[sub]
        avg = 0.0 if n == 0 else (tw / n)
        print(f"{sub} | {tw//1000}K | {n} | {(100*(avg//100)):.0f}")

    # ---- total words overall ----
    print("\nTOTAL WORDS ACROSS ALL FILES:", total_words_all)